# Trainval Scene Selection

This notebook inspects the TruckScenes v1.2-trainval metadata and builds a scene-level overview to support the later selection of approximately 200–300 scenes.

The initial screening considers sensor availability and annotation-derived vehicle motion. Additional scene factors such as distance, object type, visibility and weather will be added after the trainval metadata structure is validated.

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
import pandas as pd

from src.config import TRAINVAL_METADATA_ROOT
from src.data_io import load_json
from src.sensor_observation import build_sensor_data_df
from src.annotation_velocity import (
    build_annotation_trajectory_df,
    build_annotation_pairs,
    add_time_differences,
    add_position_changes,
    add_reference_velocity,
    add_scene_tokens,
)

In [6]:
scenes = load_json(TRAINVAL_METADATA_ROOT / "scene.json")
samples = load_json(TRAINVAL_METADATA_ROOT / "sample.json")
annotations = load_json(TRAINVAL_METADATA_ROOT / "sample_annotation.json")
instances = load_json(TRAINVAL_METADATA_ROOT / "instance.json")
categories = load_json(TRAINVAL_METADATA_ROOT / "category.json")

print("Scenes:", len(scenes))
print("Samples:", len(samples))
print("Annotations:", len(annotations))
print("Instances:", len(instances))
print("Categories:", len(categories))

Scenes: 598
Samples: 23902
Annotations: 819536
Instances: 43769
Categories: 27


In [ ]:
sensor_df = build_sensor_data_df(metadata_root=TRAINVAL_METADATA_ROOT)
scene_sensor_summary = (sensor_df.groupby(["Scene Token", "Modality"]).size().unstack(fill_value=0).reset_index())

print("Scenes with sensor metadata:", sensor_df["Scene Token"].nunique())
scene_sensor_summary.head()

Scenes with sensor metadata: 598


Modality,Scene Token,Camera,LiDAR,RADAR
0,018a60086f5e441fb09f476e55948b72,805,1206,2348
1,01f937ce4860435d986e167bbfd975be,804,1205,2387
2,028a50a9c50945a29066d6943f8feb96,804,1205,2389
3,035c713614a247feabe996f40f83a7e6,805,1208,2367
4,044c648ac12345f1aedf33c9f91cdc5a,804,1207,2222


In [8]:
trajectory_df = build_annotation_trajectory_df(samples, annotations, instances, categories,)

velocity_df = build_annotation_pairs(trajectory_df)
velocity_df = add_time_differences(velocity_df)
velocity_df = add_position_changes(velocity_df)
velocity_df = add_reference_velocity(velocity_df)
velocity_df = add_scene_tokens(velocity_df, samples)

print("Velocity observations:", len(velocity_df))
print("Scenes with velocity observations:", velocity_df["scene_token"].nunique())

Velocity observations: 775767
Scenes with velocity observations: 598


In [9]:
scene_velocity_summary = (
    velocity_df
    .groupby("scene_token")
    .agg(
        Num_Velocity_Observations=("speed_2d_mps", "count"),
        Num_Instances=("instance_token", "nunique"),
        Mean_Speed_mps=("speed_2d_mps", "mean"),
        Median_Speed_mps=("speed_2d_mps", "median"),
        Max_Speed_mps=("speed_2d_mps", "max"),
    )
    .reset_index()
    .rename(columns={"scene_token": "Scene Token"})
)

In [10]:
scene_info = pd.DataFrame([
    {
        "Scene Token": scene["token"],
        "Scene Name": scene.get("name"),
        "Description": scene.get("description"),
    }
    for scene in scenes
])

sample_counts = (
    pd.Series(
        [sample["scene_token"] for sample in samples],
        name="Scene Token"
    )
    .value_counts()
    .rename_axis("Scene Token")
    .reset_index(name="Num Samples")
)

scene_summary = (
    scene_info
    .merge(sample_counts, on="Scene Token", how="left")
    .merge(scene_sensor_summary, on="Scene Token", how="left")
    .merge(scene_velocity_summary, on="Scene Token", how="left")
)

scene_summary.head()

,Scene Token,Scene Name,Description,Num Samples,Camera,LiDAR,RADAR,Num_Velocity_Observations,Num_Instances,Mean_Speed_mps,Median_Speed_mps,Max_Speed_mps
0,d08667f00c6245e89a20240a14aafd53,scene-0044384af3d8494e913fb8b14915239e-11,weather.clear;area.parking;daytime.noon;season...,40,804,1207,2333,1765,61,0.254015,0.165285,3.172331
1,044c648ac12345f1aedf33c9f91cdc5a,scene-0044384af3d8494e913fb8b14915239e-3,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1207,2222,1245,40,0.470783,0.199320,11.077553
2,0f869ee9a3cf47c0aeb8e7c47dc23af1,scene-0044384af3d8494e913fb8b14915239e-5,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1208,2132,885,41,1.097142,0.283263,13.345920
3,9264a80a707346e79d278547fc44ec5b,scene-0044384af3d8494e913fb8b14915239e-7,weather.clear;area.parking;daytime.noon;season...,40,804,1208,2364,1639,46,0.676982,0.086720,16.973678
4,ec747ab39ab44e94924541e734e115fe,scene-0044384af3d8494e913fb8b14915239e-8,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1208,2383,292,8,0.147160,0.072447,2.131827


In [11]:
scene_summary[
    [
        "Num Samples",
        "Camera",
        "LiDAR",
        "RADAR",
        "Num_Velocity_Observations",
        "Num_Instances",
        "Mean_Speed_mps",
        "Median_Speed_mps",
        "Max_Speed_mps",
    ]
].describe().round(2)

,Num Samples,Camera,LiDAR,RADAR,Num_Velocity_Observations,Num_Instances,Mean_Speed_mps,Median_Speed_mps,Max_Speed_mps
count,598.00,598.00,598.00,598.00,598.00,598.00,598.00,598.00,598.00
mean,39.97,804.35,1205.92,2314.59,1297.27,72.10,15.39,15.90,46.54
std,0.41,0.87,5.74,76.74,812.99,41.23,9.29,12.09,32.97
min,33.00,796.00,1105.00,1940.00,215.00,6.00,0.07,0.03,0.92
25%,40.00,804.00,1205.00,2274.00,773.75,45.00,6.18,0.47,34.35
50%,40.00,804.00,1206.00,2333.50,1079.50,65.00,17.36,21.86,44.46
75%,40.00,805.00,1207.00,2374.75,1558.75,89.00,23.38,25.40,53.97
max,40.00,811.00,1209.00,2418.00,6819.00,301.00,33.77,38.08,494.41


In [12]:
scene_summary[
    (scene_summary["Camera"] == 0)
    | (scene_summary["LiDAR"] == 0)
    | (scene_summary["RADAR"] == 0)
]

,Scene Token,Scene Name,Description,Num Samples,Camera,LiDAR,RADAR,Num_Velocity_Observations,Num_Instances,Mean_Speed_mps,Median_Speed_mps,Max_Speed_mps


In [14]:
velocity_df["category"].value_counts()

category
vehicle.car                             356176
static_object.traffic_sign              174566
vehicle.truck                           107609
vehicle.trailer                          57826
movable_object.trafficcone               32684
vehicle.ego_trailer                      18084
human.pedestrian.adult                   11098
vehicle.other                             3035
vehicle.bus.rigid                         2519
vehicle.train                             1951
movable_object.barrier                    1712
vehicle.construction                      1632
vehicle.bicycle                           1570
vehicle.motorcycle                        1323
movable_object.pushable_pullable          1027
human.pedestrian.construction_worker       892
animal                                     822
vehicle.emergency.ambulance                307
human.pedestrian.child                     271
movable_object.debris                      262
human.pedestrian.personal_mobility         142
stat

In [13]:
velocity_df.nlargest(
    20,
    "speed_2d_mps"
)[
    [
        "scene_token",
        "instance_token",
        "category",
        "dt_s",
        "displacement_2d",
        "speed_2d_mps",
    ]
]

,scene_token,instance_token,category,dt_s,displacement_2d,speed_2d_mps
292721,e1521b93d0be4ad6880bf45b8554a8fd,c796d9c9932a4cb6896ce6f0f89c435b,vehicle.train,0.499555,246.984810,494.409645
292846,0be0c29709ac4aca8eb2327d4eae6bd1,1d25a3e39b714cc790439d789f5b470e,vehicle.train,0.500782,195.508644,390.406692
292762,e1521b93d0be4ad6880bf45b8554a8fd,e462b20bdd31460d9477596e5b982409,vehicle.train,0.500195,158.620258,317.116841
566416,d149f8b6b8794e6bb1a0981fa4820886,c0043e00c5d340b1bdb2f707e8ca36dd,vehicle.car,0.499825,155.654176,311.417348
292522,e1521b93d0be4ad6880bf45b8554a8fd,1225c82d652449b0be43850b7a15f33f,vehicle.train,0.499834,143.621803,287.339003
292525,e1521b93d0be4ad6880bf45b8554a8fd,1225c82d652449b0be43850b7a15f33f,vehicle.train,0.500195,140.167196,280.225105
292863,0be0c29709ac4aca8eb2327d4eae6bd1,1d25a3e39b714cc790439d789f5b470e,vehicle.train,0.500515,137.922370,275.560912
292861,0be0c29709ac4aca8eb2327d4eae6bd1,1d25a3e39b714cc790439d789f5b470e,vehicle.train,0.500040,99.215489,198.415104
292725,e1521b93d0be4ad6880bf45b8554a8fd,c796d9c9932a4cb6896ce6f0f89c435b,vehicle.train,0.499683,95.069467,190.259559
773005,d6930309039e4a1ab75ea14cc37f6a2a,a9666093f04945668cc25a1e8a7cd00c,vehicle.car,0.500296,81.833428,163.570022


In [15]:
scene_info[["Scene Name", "Description"]].head(20)

,Scene Name,Description
0,scene-0044384af3d8494e913fb8b14915239e-11,weather.clear;area.parking;daytime.noon;season...
1,scene-0044384af3d8494e913fb8b14915239e-3,weather.clear;area.terminal;daytime.noon;seaso...
2,scene-0044384af3d8494e913fb8b14915239e-5,weather.clear;area.terminal;daytime.noon;seaso...
3,scene-0044384af3d8494e913fb8b14915239e-7,weather.clear;area.parking;daytime.noon;season...
4,scene-0044384af3d8494e913fb8b14915239e-8,weather.clear;area.terminal;daytime.noon;seaso...
5,scene-0afa5cb3c6d34a998a7da1c6206511c7-1,weather.rain;area.highway;daytime.noon;season....
6,scene-0afa5cb3c6d34a998a7da1c6206511c7-2,weather.rain;area.highway;daytime.noon;season....
7,scene-0afa5cb3c6d34a998a7da1c6206511c7-3,weather.rain;area.highway;daytime.noon;season....
8,scene-0afa5cb3c6d34a998a7da1c6206511c7-4,weather.rain;area.highway;daytime.noon;season....
9,scene-0afa5cb3c6d34a998a7da1c6206511c7-5,weather.rain;area.highway;daytime.noon;season....


In [16]:
for description in scene_info["Description"].dropna().unique()[:20]:
    print(description)

weather.clear;area.parking;daytime.noon;season.summer;lighting.illuminated;structure.regular;construction.unchanged
weather.clear;area.terminal;daytime.noon;season.autumn;lighting.illuminated;structure.regular;construction.unchanged
weather.clear;area.terminal;daytime.noon;season.summer;lighting.illuminated;structure.regular;construction.unchanged
weather.clear;area.parking;daytime.noon;season.autumn;lighting.illuminated;structure.regular;construction.unchanged
weather.rain;area.highway;daytime.noon;season.autumn;lighting.glare;structure.regular;construction.unchanged
weather.rain;area.highway;daytime.noon;season.autumn;lighting.glare;structure.underpass;construction.unchanged
weather.rain;area.highway;daytime.noon;season.autumn;lighting.dark;structure.regular;construction.unchanged
weather.clear;area.highway;daytime.noon;season.summer;lighting.illuminated;structure.regular;construction.unchanged
weather.clear;area.highway;daytime.noon;season.summer;lighting.illuminated;structure.under

In [17]:
def parse_scene_description(description):
    values = {}

    if not isinstance(description, str):
        return values

    for item in description.split(";"):
        if "." in item:
            key, value = item.split(".", 1)
            values[key] = value

    return values


scene_context = (
    scene_info["Description"]
    .apply(parse_scene_description)
    .apply(pd.Series)
)

scene_info = pd.concat(
    [scene_info, scene_context],
    axis=1,
)

scene_info.head()

,Scene Token,Scene Name,Description,weather,area,daytime,season,lighting,structure,construction
0,d08667f00c6245e89a20240a14aafd53,scene-0044384af3d8494e913fb8b14915239e-11,weather.clear;area.parking;daytime.noon;season...,clear,parking,noon,summer,illuminated,regular,unchanged
1,044c648ac12345f1aedf33c9f91cdc5a,scene-0044384af3d8494e913fb8b14915239e-3,weather.clear;area.terminal;daytime.noon;seaso...,clear,terminal,noon,autumn,illuminated,regular,unchanged
2,0f869ee9a3cf47c0aeb8e7c47dc23af1,scene-0044384af3d8494e913fb8b14915239e-5,weather.clear;area.terminal;daytime.noon;seaso...,clear,terminal,noon,summer,illuminated,regular,unchanged
3,9264a80a707346e79d278547fc44ec5b,scene-0044384af3d8494e913fb8b14915239e-7,weather.clear;area.parking;daytime.noon;season...,clear,parking,noon,autumn,illuminated,regular,unchanged
4,ec747ab39ab44e94924541e734e115fe,scene-0044384af3d8494e913fb8b14915239e-8,weather.clear;area.terminal;daytime.noon;seaso...,clear,terminal,noon,summer,illuminated,regular,unchanged


In [18]:
for column in [
    "weather",
    "area",
    "daytime",
    "season",
    "lighting",
    "structure",
    "construction",
]:
    print(f"\n{column}")
    print(scene_info[column].value_counts())


weather
weather
clear            264
overcast         178
rain             119
snow              23
fog               12
other_weather      2
Name: count, dtype: int64

area
area
highway        441
terminal        54
rural           52
city            35
residential     12
parking          3
other_area       1
Name: count, dtype: int64

daytime
daytime
noon       319
morning    228
evening     51
Name: count, dtype: int64

season
season
summer    272
autumn    225
winter    101
Name: count, dtype: int64

lighting
lighting
illuminated       480
twilight           44
dark               38
glare              34
other_lighting      2
Name: count, dtype: int64

structure
structure
regular      385
underpass    109
overpass      64
bridge        30
tunnel        10
Name: count, dtype: int64

construction
construction
unchanged    577
roadworks     21
Name: count, dtype: int64


In [19]:
vehicle_velocity_df = velocity_df[
    velocity_df["category"].str.startswith("vehicle.")
].copy()

vehicle_velocity_df["category"].value_counts()

category
vehicle.car                    356176
vehicle.truck                  107609
vehicle.trailer                 57826
vehicle.ego_trailer             18084
vehicle.other                    3035
vehicle.bus.rigid                2519
vehicle.train                    1951
vehicle.construction             1632
vehicle.bicycle                  1570
vehicle.motorcycle               1323
vehicle.emergency.ambulance       307
vehicle.emergency.police           76
vehicle.bus.bendy                  15
Name: count, dtype: int64

In [20]:
vehicle_velocity_df["speed_2d_mps"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

count    552123.000000
mean         18.502751
std          13.034038
min           0.000000
50%          22.280768
90%          34.107047
95%          36.963019
99%          43.664428
max         494.409645
Name: speed_2d_mps, dtype: float64

In [21]:
vehicle_velocity_df.groupby("category")["speed_2d_mps"].agg(
    ["count", "median", "mean", "max"]
).round(2)

,count,median,mean,max
category,,,,
vehicle.bicycle,1570,2.00,2.81,35.72
vehicle.bus.bendy,15,0.18,0.19,0.42
vehicle.bus.rigid,2519,0.42,9.11,35.17
vehicle.car,356176,24.77,21.03,311.42
vehicle.construction,1632,0.30,0.70,11.66
vehicle.ego_trailer,18084,18.92,15.71,30.62
vehicle.emergency.ambulance,307,28.01,28.68,41.87
vehicle.emergency.police,76,28.44,24.39,34.12
vehicle.motorcycle,1323,28.18,21.52,46.25


In [23]:
scene_vehicle_summary = (
    vehicle_velocity_df
    .groupby("scene_token")
    .agg(
        Num_Vehicle_Observations=("speed_2d_mps", "count"),
        Num_Vehicles=("instance_token", "nunique"),
        Mean_Vehicle_Speed_mps=("speed_2d_mps", "mean"),
        Median_Vehicle_Speed_mps=("speed_2d_mps", "median"),
        P95_Vehicle_Speed_mps=(
            "speed_2d_mps",
            lambda x: x.quantile(0.95),
        ),
    )
    .reset_index()
    .rename(columns={"scene_token": "Scene Token"})
)

scene_vehicle_summary.head()

,Scene Token,Num_Vehicle_Observations,Num_Vehicles,Mean_Vehicle_Speed_mps,Median_Vehicle_Speed_mps,P95_Vehicle_Speed_mps
0,018a60086f5e441fb09f476e55948b72,938,56,30.482390,31.956725,45.811574
1,01f937ce4860435d986e167bbfd975be,630,45,31.019907,29.915136,41.380361
2,028a50a9c50945a29066d6943f8feb96,1007,67,24.135325,24.607567,31.932011
3,035c713614a247feabe996f40f83a7e6,169,10,19.975495,18.430449,29.322059
4,044c648ac12345f1aedf33c9f91cdc5a,1083,32,0.463311,0.190544,2.417610


In [24]:
scene_summary = (
    scene_summary
    .drop(
        columns=[
            "Num_Velocity_Observations",
            "Num_Instances",
            "Mean_Speed_mps",
            "Median_Speed_mps",
            "Max_Speed_mps",
        ],
        errors="ignore",
    )
    .merge(
        scene_vehicle_summary,
        on="Scene Token",
        how="left",
    )
    .merge(
        scene_info[
            [
                "Scene Token",
                "weather",
                "area",
                "daytime",
                "season",
                "lighting",
                "structure",
                "construction",
            ]
        ],
        on="Scene Token",
        how="left",
    )
)

In [25]:
annotations[0]

{'token': '598cffe6e9794508b560522a760f1b7c',
 'sample_token': '1bb41855cb724ae6980bababbd1865e2',
 'instance_token': '0472a737bb8848da835677cb662225f2',
 'visibility_token': 'f95237cfe4e0480c899fab6d5e5a235c',
 'attribute_tokens': ['eefd82c9b2f144e2b1c61a432254cff2'],
 'translation': [572005.1694430078, 5367793.52429555, 591.4399098568119],
 'size': [1.607876, 4.923336, 1.84361],
 'rotation': [0.919380738699952,
  -1.734723475976807e-18,
  0.0,
  0.3933688565551808],
 'prev': '',
 'next': 'eaa4d08b7aa24930a53f1725faeeabe1',
 'num_lidar_pts': 14,
 'num_radar_pts': 0}

In [26]:
scene_summary = (
    scene_summary
    .drop(
        columns=[
            "Num_Velocity_Observations",
            "Num_Instances",
            "Mean_Speed_mps",
            "Median_Speed_mps",
            "Max_Speed_mps",
        ],
        errors="ignore",
    )
    .merge(
        scene_vehicle_summary,
        on="Scene Token",
        how="left",
    )
    .merge(
        scene_info[
            [
                "Scene Token",
                "weather",
                "area",
                "daytime",
                "season",
                "lighting",
                "structure",
                "construction",
            ]
        ],
        on="Scene Token",
        how="left",
    )
)

scene_summary.head()

,Scene Token,Scene Name,Description,Num Samples,Camera,LiDAR,RADAR,Num_Vehicle_Observations_x,Num_Vehicles_x,Mean_Vehicle_Speed_mps_x,...,Mean_Vehicle_Speed_mps_y,Median_Vehicle_Speed_mps_y,P95_Vehicle_Speed_mps_y,weather_y,area_y,daytime_y,season_y,lighting_y,structure_y,construction_y
0,d08667f00c6245e89a20240a14aafd53,scene-0044384af3d8494e913fb8b14915239e-11,weather.clear;area.parking;daytime.noon;season...,40,804,1207,2333,1205,43,0.276106,...,0.276106,0.172696,0.721714,clear,parking,noon,summer,illuminated,regular,unchanged
1,044c648ac12345f1aedf33c9f91cdc5a,scene-0044384af3d8494e913fb8b14915239e-3,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1207,2222,1083,32,0.463311,...,0.463311,0.190544,2.417610,clear,terminal,noon,autumn,illuminated,regular,unchanged
2,0f869ee9a3cf47c0aeb8e7c47dc23af1,scene-0044384af3d8494e913fb8b14915239e-5,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1208,2132,661,30,1.316146,...,1.316146,0.311493,5.014833,clear,terminal,noon,summer,illuminated,regular,unchanged
3,9264a80a707346e79d278547fc44ec5b,scene-0044384af3d8494e913fb8b14915239e-7,weather.clear;area.parking;daytime.noon;season...,40,804,1208,2364,1346,37,0.804293,...,0.804293,0.093966,4.508632,clear,parking,noon,autumn,illuminated,regular,unchanged
4,ec747ab39ab44e94924541e734e115fe,scene-0044384af3d8494e913fb8b14915239e-8,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1208,2383,234,6,0.099166,...,0.099166,0.060709,0.338048,clear,terminal,noon,summer,illuminated,regular,unchanged


In [30]:
OUTPUT_PATH = PROJECT_ROOT / "results" / "scene_selection" / "trainval_scene_summary.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
scene_summary.to_csv(OUTPUT_PATH, index=False)

In [31]:
scene_summary = scene_summary.loc[
    :, ~scene_summary.columns.str.endswith("_y")
].copy()

scene_summary.columns = [
    column[:-2] if column.endswith("_x") else column
    for column in scene_summary.columns
]

scene_summary.head()

,Scene Token,Scene Name,Description,Num Samples,Camera,LiDAR,RADAR,Num_Vehicle_Observations,Num_Vehicles,Mean_Vehicle_Speed_mps,Median_Vehicle_Speed_mps,P95_Vehicle_Speed_mps,weather,area,daytime,season,lighting,structure,construction
0,d08667f00c6245e89a20240a14aafd53,scene-0044384af3d8494e913fb8b14915239e-11,weather.clear;area.parking;daytime.noon;season...,40,804,1207,2333,1205,43,0.276106,0.172696,0.721714,clear,parking,noon,summer,illuminated,regular,unchanged
1,044c648ac12345f1aedf33c9f91cdc5a,scene-0044384af3d8494e913fb8b14915239e-3,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1207,2222,1083,32,0.463311,0.190544,2.417610,clear,terminal,noon,autumn,illuminated,regular,unchanged
2,0f869ee9a3cf47c0aeb8e7c47dc23af1,scene-0044384af3d8494e913fb8b14915239e-5,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1208,2132,661,30,1.316146,0.311493,5.014833,clear,terminal,noon,summer,illuminated,regular,unchanged
3,9264a80a707346e79d278547fc44ec5b,scene-0044384af3d8494e913fb8b14915239e-7,weather.clear;area.parking;daytime.noon;season...,40,804,1208,2364,1346,37,0.804293,0.093966,4.508632,clear,parking,noon,autumn,illuminated,regular,unchanged
4,ec747ab39ab44e94924541e734e115fe,scene-0044384af3d8494e913fb8b14915239e-8,weather.clear;area.terminal;daytime.noon;seaso...,40,804,1208,2383,234,6,0.099166,0.060709,0.338048,clear,terminal,noon,summer,illuminated,regular,unchanged


In [32]:
TARGET_SCENES = 250

scene_summary["Eligible"] = (
    (scene_summary["Num_Vehicles"] >= 10)
    & (scene_summary["Num_Vehicle_Observations"] >= 200)
)

speed_labels = ["Q1_low", "Q2", "Q3", "Q4_high"]

scene_summary["Selection_Speed_Group"] = pd.qcut(
    scene_summary["Median_Vehicle_Speed_mps"],
    q=4,
    labels=speed_labels,
)

print("Total scenes:", len(scene_summary))
print("Eligible scenes:", scene_summary["Eligible"].sum())
print("Context combinations:", scene_summary["Description"].nunique())

Total scenes: 598
Eligible scenes: 579
Context combinations: 154


In [33]:
eligible_representatives = (
    scene_summary[scene_summary["Eligible"]]
    .sort_values(
        ["Num_Vehicle_Observations", "Num_Vehicles"],
        ascending=False,
    )
    .drop_duplicates("Description")
)

covered_descriptions = set(
    eligible_representatives["Description"]
)

missing_descriptions = (
    set(scene_summary["Description"])
    - covered_descriptions
)

fallback_representatives = (
    scene_summary[
        scene_summary["Description"].isin(missing_descriptions)
    ]
    .sort_values(
        ["Num_Vehicle_Observations", "Num_Vehicles"],
        ascending=False,
    )
    .drop_duplicates("Description")
)

selected = pd.concat(
    [
        eligible_representatives,
        fallback_representatives,
    ],
    ignore_index=True,
)

selected["Selection_Reason"] = "context_representative"

print("Initial selected scenes:", len(selected))

Initial selected scenes: 154


In [34]:
base_quota = TARGET_SCENES // len(speed_labels)
remainder = TARGET_SCENES % len(speed_labels)

speed_quota = {
    label: base_quota + (i < remainder)
    for i, label in enumerate(speed_labels)
}

selected_tokens = set(selected["Scene Token"])

additional_scenes = []

for label in speed_labels:
    current_count = (
        selected["Selection_Speed_Group"] == label
    ).sum()

    needed = speed_quota[label] - current_count

    if needed <= 0:
        continue

    pool = scene_summary[
        (scene_summary["Eligible"])
        & (scene_summary["Selection_Speed_Group"] == label)
        & (~scene_summary["Scene Token"].isin(selected_tokens))
    ]

    sample = pool.sample(
        n=needed,
        random_state=42,
    ).copy()

    sample["Selection_Reason"] = "speed_balance"

    additional_scenes.append(sample)

selected_scenes = pd.concat(
    [selected] + additional_scenes,
    ignore_index=True,
)

print("Selected scenes:", len(selected_scenes))
print(
    selected_scenes[
        "Selection_Speed_Group"
    ].value_counts().sort_index()
)

Selected scenes: 250
Selection_Speed_Group
Q1_low     63
Q2         63
Q3         62
Q4_high    62
Name: count, dtype: int64


In [35]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "results"
    / "scene_selection"
    / "selected_scenes.csv"
)

selected_scenes.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: f:\CITS3200\Boeing-RADAR-Autonomous-\results\scene_selection\selected_scenes.csv
